In [ ]:
# Cell 1 — Setup (now uses image.png already in the Colab file list)
# ------------------------------------------------------------
# Assumes you already uploaded a file named: image.png
# (Left sidebar → Files → upload image.png)

!pip -q install opencv-python ipywidgets

import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams["figure.dpi"] = 140

def show(img, title=""):
    plt.figure(figsize=(5,4))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        plt.imshow(img)
    plt.axis("off")
    plt.title(title)
    plt.show()

def show_row(imgs, titles):
    plt.figure(figsize=(4*len(imgs), 4))
    for i,(im,t) in enumerate(zip(imgs,titles)):
        plt.subplot(1, len(imgs), i+1)
        if im.ndim == 2:
            plt.imshow(im, cmap="gray", vmin=0, vmax=255)
        else:
            plt.imshow(im)
        plt.axis("off")
        plt.title(t)
    plt.tight_layout()
    plt.show()

# --- Load the already-uploaded file ---
path = "image.png"  # must exist in the Colab working directory
bgr = cv2.imread(path, cv2.IMREAD_COLOR)
if bgr is None:
    raise FileNotFoundError(
        "Couldn't find image.png. Upload it in the Colab left sidebar (Files) and try again."
    )

img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)        # for correct display
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)      # for filtering demos

show(img, "Loaded: image.png (RGB)")

In [ ]:
# Cell 2 — (Optional) Reload image.png if you replace it
# ------------------------------------------------------------
# If you upload a NEW image.png (same filename) and want to refresh it,
# run this cell again.

bgr = cv2.imread("image.png", cv2.IMREAD_COLOR)
if bgr is None:
    raise FileNotFoundError("Couldn't find image.png. Make sure it's uploaded.")
img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

show(img, "Reloaded: image.png (RGB)")

In [ ]:
# Cell 3 — 1) What is a filter? (pixels + kernels) + 3 clear examples
# ------------------------------------------------------------
# A (convolution) filter replaces each pixel with a weighted sum of nearby pixels.
# Those weights are stored in a small matrix called a "kernel".

# --- BLUR (7x7 box blur) ---
# What it does: replaces each pixel with the average of a 7x7 neighborhood.
# Effect: reduces noise and fine detail.
blur_k = np.ones((7,7), np.float32) / 49.0

# --- SHARPEN (3x3) ---
# What it does: boosts the center pixel and subtracts neighboring pixels.
# Effect: increases contrast around edges (makes edges stand out more).
sharpen_k = np.array([[ 0,-1, 0],
                      [-1, 5,-1],
                      [ 0,-1, 0]], np.float32)

# --- LAPLACIAN (edge detection, 3x3) ---
# What it does: measures rapid changes in brightness (a second-derivative operator).
# Effect: highlights boundaries where intensity changes quickly.
laplacian_k = np.array([[ 0, 1, 0],
                        [ 1,-4, 1],
                        [ 0, 1, 0]], np.float32)

# Apply kernels to the grayscale image
blur = cv2.filter2D(gray, -1, blur_k)
sharp = cv2.filter2D(gray, -1, sharpen_k)

# Laplacian can produce negative values, so compute in float, take absolute value, then scale to 0..255 for display
lap = cv2.filter2D(gray, cv2.CV_32F, laplacian_k)
lap = np.abs(lap)
lap = (lap / (lap.max() + 1e-6) * 255).astype(np.uint8)

show_row(
    [gray, blur, sharp, lap],
    ["Original (grayscale)",
     "Box Blur (7x7)",
     "Sharpen (3x3)",
     "Laplacian Edge Detection (3x3)"]
)

print("Blur kernel (7x7):\n", blur_k)
print("\nSharpen kernel (3x3):\n", sharpen_k)
print("\nLaplacian kernel (3x3):\n", laplacian_k)

In [ ]:
# Cell 4 — 2) Convolution playground (simple controls)
# ------------------------------------------------------------
# This section lets you change a filter parameter and immediately see what happens.
#
# We provide three options:
# - Blur (Gaussian blur)
# - Sharpen (unsharp mask)
# - Edge Detection (Canny)
#
# One slider:
# - k = the kernel size (odd numbers: 1, 3, 5, ...). It controls the neighborhood size.

filter_choice = widgets.Dropdown(
    options=["Blur", "Sharpen", "Edges"],
    value="Blur",
    description="Filter:"
)

k_slider = widgets.IntSlider(
    value=5, min=1, max=31, step=2,
    description="k:",
    continuous_update=False
)

def playground(choice, k):
    if choice == "Blur":
        # What it does: replaces each pixel with a weighted average of nearby pixels.
        # Effect: reduces noise and fine detail.
        out = cv2.GaussianBlur(gray, (k, k), 0)

    elif choice == "Sharpen":
        # What it does: adds back high-frequency detail by subtracting a blurred version.
        # Effect: increases contrast around edges (makes edges stand out more).
        blur = cv2.GaussianBlur(gray, (k, k), 0)
        out_f = cv2.addWeighted(gray.astype(np.float32), 1.5,
                                blur.astype(np.float32), -0.5, 0)
        out = np.clip(out_f, 0, 255).astype(np.uint8)

    else:  # "Edges"
        # What it does: detects boundaries where intensity changes quickly.
        # Effect: produces a binary image showing edges.
        g = cv2.GaussianBlur(gray, (k, k), 0)
        out = cv2.Canny(g, 60, 140)

    show_row([gray, out], ["Input (grayscale)", f"Output: {choice} (k={k})"])

display(
    filter_choice,
    k_slider,
    widgets.interactive_output(playground, {"choice": filter_choice, "k": k_slider})
)

In [ ]:
# Cell 5 — 3) Add noise and remove it
# ------------------------------------------------------------
# Noise = unwanted random variation in pixel values.
#
# We add two types of noise:
# - Gaussian noise: adds small random values to many pixels.
# - Salt & pepper noise: flips some pixels to black (0) or white (255).
#
# Then we remove noise with two filters:
# - Gaussian blur:
#   What it does: replaces each pixel with a weighted average of nearby pixels.
#   Effect: reduces Gaussian noise, but also reduces fine detail.
# - Median filter:
#   What it does: replaces each pixel with the median value in its neighborhood.
#   Effect: removes salt & pepper noise well and preserves edges better than averaging.

noise_kind = widgets.Dropdown(
    options=["Gaussian", "Salt & Pepper"],
    value="Gaussian",
    description="Noise:"
)

amount = widgets.IntSlider(
    value=20, min=0, max=80, step=1,
    description="Amount:",
    continuous_update=False
)

k_denoise = widgets.IntSlider(
    value=5, min=1, max=15, step=2,
    description="k:",
    continuous_update=False
)

def add_gaussian(x, sigma):
    # Add random values drawn from a normal distribution (mean 0, std = sigma).
    n = np.random.normal(0, sigma, x.shape).astype(np.float32)
    y = x.astype(np.float32) + n
    return np.clip(y, 0, 255).astype(np.uint8)

def add_sp(x, amount):
    # Flip some pixels to 0 or 255.
    y = x.copy()
    h, w = y.shape
    p = amount / 100.0
    num = int(p * h * w * 0.15)  # scaling factor for a reasonable demo
    ys = np.random.randint(0, h, num); xs = np.random.randint(0, w, num); y[ys, xs] = 255
    ys = np.random.randint(0, h, num); xs = np.random.randint(0, w, num); y[ys, xs] = 0
    return y

def noise_demo(kind, amt, k):
    # 1) Make a noisy image
    noisy = add_gaussian(gray, amt) if kind == "Gaussian" else add_sp(gray, amt)

    # 2) Denoise with Gaussian blur (weighted averaging)
    den_gauss = cv2.GaussianBlur(noisy, (k, k), 0)

    # 3) Denoise with median filter (median in neighborhood)
    den_med = cv2.medianBlur(noisy, k)

    # NOTE: To avoid title overlap, we keep titles short and rely on the comments above.
    show_row(
        [gray, noisy, den_gauss, den_med],
        ["Original",
         f"Noisy ({kind})",
         f"Gaussian blur (k={k})",
         f"Median filter (k={k})"]
    )

display(
    noise_kind,
    amount,
    k_denoise,
    widgets.interactive_output(noise_demo, {"kind": noise_kind, "amt": amount, "k": k_denoise})
)

In [ ]:
# Cell 6 — 4) Edge detection
# ------------------------------------------------------------
# Edges are places where brightness changes quickly (object boundaries).
#
# Two edge detectors:
# - Sobel:
#   What it does: measures brightness change left-right and up-down (image gradients).
#   Effect: produces an “edge strength” image (stronger values = stronger edges).
# - Canny:
#   What it does: finds edges by looking for strong intensity changes and linking them into curves.
#   Effect: produces a binary image (edges vs. non-edges).
#
# We also apply a blur first:
#   What it does: averages nearby pixels (smoothing).
#   Effect: reduces noise so we detect fewer spurious edges.

method = widgets.Dropdown(
    options=["Sobel", "Canny"],
    value="Canny",
    description="Method:"
)

k_blur = widgets.IntSlider(
    value=5, min=1, max=21, step=2,
    description="Blur k:",
    continuous_update=False
)

def edges_demo(m, k):
    # Pre-blur to reduce noise-induced edges
    g = cv2.GaussianBlur(gray, (k, k), 0)

    if m == "Canny":
        out = cv2.Canny(g, 60, 140)
    else:
        gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx*gx + gy*gy)
        out = (mag / (mag.max() + 1e-6) * 255).astype(np.uint8)

    # Keep titles short to avoid overlap
    show_row([gray, g, out], ["Original", f"Blur (k={k})", m])

display(
    method,
    k_blur,
    widgets.interactive_output(edges_demo, {"m": method, "k": k_blur})
)

In [ ]:
# Cell 7 — 5) Sketched avatar (filter pipeline with toggles)
# ------------------------------------------------------------
# This section combines the filters you've already seen into one pipeline.
# You can switch which denoiser and which edge detector are used.
#
# Pipeline steps:
#
# 1) Grayscale (already stored in `gray`)
#   What it does: represents the image using a single brightness value per pixel.
#   Effect: simplifies the problem so we only work with intensity.
#
# 2) Denoise (choose one)
#   - Gaussian blur
#     What it does: replaces each pixel with a weighted average of nearby pixels.
#     Effect: reduces Gaussian-like noise and fine texture, but softens edges.
#   - Median filter
#     What it does: replaces each pixel with the median value in its neighborhood.
#     Effect: removes salt & pepper noise well and keeps edges sharper than averaging.
#   - None
#     What it does: skips denoising.
#     Effect: edges may become noisier, but fine details are preserved.
#
# 3) Edge detection (choose one)
#   - Canny
#     What it does: finds strong intensity changes and links them into thin edge curves.
#     Effect: produces a clean binary edge map (edge / no-edge).
#   - Sobel
#     What it does: measures brightness change left-right and up-down (gradients).
#     Effect: produces an edge-strength image; we threshold it to get a binary edge map.
#   - Laplacian
#     What it does: measures rapid changes in brightness (second derivative).
#     Effect: highlights boundaries strongly; we threshold it to get a binary edge map.
#
# 4) Invert
#   What it does: converts white edges on black background into black lines on white paper.
#   Effect: makes the result look like ink/pencil lines.
#
# 5) Thicken (optional)
#   What it does: slightly thickens black lines by eroding the white background.
#   Effect: gives heavier strokes.

denoise_choice = widgets.Dropdown(
    options=["Gaussian blur", "Median filter", "None"],
    value="Gaussian blur",
    description="Denoise:"
)

edge_choice = widgets.Dropdown(
    options=["Canny", "Sobel", "Laplacian"],
    value="Canny",
    description="Edges:"
)

k_sketch = widgets.IntSlider(
    value=7, min=1, max=21, step=2,
    description="k:",
    continuous_update=False
)

# Parameters for Canny (only used when Edges = Canny)
canny_t1 = widgets.IntSlider(value=60, min=0, max=255, step=1, description="T1:", continuous_update=False)
canny_t2 = widgets.IntSlider(value=140, min=0, max=255, step=1, description="T2:", continuous_update=False)

# Threshold used to turn Sobel/Laplacian "edge strength" into a binary edge map
edge_thresh = widgets.IntSlider(value=60, min=0, max=255, step=1, description="Thr:", continuous_update=False)

thicken = widgets.IntSlider(
    value=0, min=0, max=3, step=1,
    description="Thicken:",
    continuous_update=False
)

def sketch_pipeline(denoise, edge, k, t1, t2, thr, thick):
    # 1) Denoise
    if denoise == "Gaussian blur":
        smooth = cv2.GaussianBlur(gray, (k, k), 0) if k > 1 else gray
    elif denoise == "Median filter":
        smooth = cv2.medianBlur(gray, k) if k > 1 else gray
    else:
        smooth = gray

    # 2) Edge detection
    if edge == "Canny":
        edges = cv2.Canny(smooth, t1, t2)

    elif edge == "Sobel":
        gx = cv2.Sobel(smooth, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx * gx + gy * gy)
        mag = (mag / (mag.max() + 1e-6) * 255).astype(np.uint8)
        edges = ((mag > thr).astype(np.uint8) * 255)

    else:  # Laplacian
        lap = cv2.Laplacian(smooth, cv2.CV_32F, ksize=3)
        lap = np.abs(lap)
        lap = (lap / (lap.max() + 1e-6) * 255).astype(np.uint8)
        edges = ((lap > thr).astype(np.uint8) * 255)

    # 3) Invert to get black strokes on white
    sketch_img = 255 - edges

    # 4) Optional thickening
    if thick > 0:
        kernel = np.ones((3, 3), np.uint8)
        sketch_img = cv2.erode(sketch_img, kernel, iterations=thick)

    # Keep titles short to avoid overlap
    show_row(
        [img, smooth, edges, sketch_img],
        ["Original", "After denoise", "Edges", "Sketch"]
    )
    print(f"Denoise: {denoise} | Edge detector: {edge} | k={k} | Thicken={thick}")

ui = widgets.VBox([
    widgets.HBox([denoise_choice, edge_choice]),
    widgets.HBox([k_sketch, thicken]),
    widgets.HBox([canny_t1, canny_t2, edge_thresh]),
])

display(
    ui,
    widgets.interactive_output(
        sketch_pipeline,
        {"denoise": denoise_choice, "edge": edge_choice, "k": k_sketch,
         "t1": canny_t1, "t2": canny_t2, "thr": edge_thresh, "thick": thicken}
    )
)

In [ ]:
# Cell 8 — Color options for the sketch (line color + background color)
# ------------------------------------------------------------
# What it does: turns the black-and-white sketch into a colored sketch by
# mapping “line pixels” to a line color and “paper pixels” to a background color.
# Effect: students can make different “ink” and “paper” styles (e.g., blue ink on beige paper).

import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

def hex_to_rgb01(hex_color):
    """'#RRGGBB' -> (r,g,b) in 0..1"""
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16) / 255.0
    g = int(h[2:4], 16) / 255.0
    b = int(h[4:6], 16) / 255.0
    return np.array([r, g, b], dtype=np.float32)

def compute_sketch_gray_current():
    """Recompute the sketch (grayscale) using the CURRENT Cell 7 toggle/slider values."""
    # 1) Denoise
    k = k_sketch.value
    denoise = denoise_choice.value
    edge = edge_choice.value
    t1, t2 = canny_t1.value, canny_t2.value
    thr = edge_thresh.value
    thick = thicken.value

    if denoise == "Gaussian blur":
        smooth = cv2.GaussianBlur(gray, (k, k), 0) if k > 1 else gray
    elif denoise == "Median filter":
        smooth = cv2.medianBlur(gray, k) if k > 1 else gray
    else:
        smooth = gray

    # 2) Edge detection
    if edge == "Canny":
        edges = cv2.Canny(smooth, t1, t2)
    elif edge == "Sobel":
        gx = cv2.Sobel(smooth, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(smooth, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx * gx + gy * gy)
        mag = (mag / (mag.max() + 1e-6) * 255).astype(np.uint8)
        edges = ((mag > thr).astype(np.uint8) * 255)
    else:  # Laplacian
        lap = cv2.Laplacian(smooth, cv2.CV_32F, ksize=3)
        lap = np.abs(lap)
        lap = (lap / (lap.max() + 1e-6) * 255).astype(np.uint8)
        edges = ((lap > thr).astype(np.uint8) * 255)

    # 3) Invert -> black lines on white
    sketch_gray = 255 - edges

    # 4) Optional thicken
    if thick > 0:
        kernel = np.ones((3, 3), np.uint8)
        sketch_gray = cv2.erode(sketch_gray, kernel, iterations=thick)

    return sketch_gray

line_color = widgets.ColorPicker(
    value="#000000",
    description="Line:",
)

bg_color = widgets.ColorPicker(
    value="#ffffff",
    description="Background:",
)

blend_mode = widgets.Checkbox(
    value=True,
    description="Smooth lines (blend)",
)

color_out = widgets.Output()

def render_colored(line_hex, bg_hex, blend):
    sketch_gray = compute_sketch_gray_current()

    # alpha = how much “ink” to apply at each pixel
    # - If blend=True: use grayscale intensity for smooth transitions
    # - If blend=False: hard threshold for a crisp binary look
    if blend:
        alpha = 1.0 - (sketch_gray.astype(np.float32) / 255.0)   # 0 on white, 1 on black
    else:
        alpha = (sketch_gray < 128).astype(np.float32)           # 0/1 mask

    alpha = alpha[..., None]  # (H,W,1)

    ink = hex_to_rgb01(line_hex)[None, None, :]  # (1,1,3)
    paper = hex_to_rgb01(bg_hex)[None, None, :]  # (1,1,3)

    colored = paper * (1.0 - alpha) + ink * alpha
    colored = np.clip(colored * 255.0, 0, 255).astype(np.uint8)

    color_out.clear_output()
    with color_out:
        show_row([img, colored], ["Original", "Colored sketch"])

display(
    widgets.HBox([line_color, bg_color, blend_mode]),
    color_out
)

# Initial render + auto-update when controls change
def _update(_=None):
    render_colored(line_color.value, bg_color.value, blend_mode.value)

line_color.observe(_update, names="value")
bg_color.observe(_update, names="value")
blend_mode.observe(_update, names="value")
_update()